# 01 Data And Subsets
Rohdaten laden, mentions bauen, Stage-Subsets erzeugen, QC ausgeben.

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = None  # Optional override. Default: auto from notebook 00 latest_run.json


ROOT: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation


In [2]:
import pandas as pd

from src.common.config import (
    load_notebook_context,
    build_run_dirs,
    resolve_run_id,
    write_run_consistency,
)
from src.data.prepare_lspo import prepare_lspo_mentions
from src.data.prepare_ads import prepare_ads_mentions
from src.common.subset_builder import build_stage_subset, write_subset_manifest
from src.common.io_schema import save_parquet
from src.common.run_report import summarize_block_distribution

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
RUN = CTX["RUN"]

latest_context_path = Path(ART["metrics_dir"]) / "latest_run.json"
RUN_ID = resolve_run_id(RUN_ID, latest_context_path, allow_placeholder=False)
RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
for key in ["metrics", "subset_cache", "subset_manifests", "interim"]:
    RUN_DIRS[key].mkdir(parents=True, exist_ok=True)

write_run_consistency(
    run_id=RUN_ID,
    run_stage=RUN_STAGE,
    run_dirs=RUN_DIRS,
    output_path=RUN_DIRS["metrics"] / "01_run_consistency.json",
    extras={"notebook": "01_data_and_subsets", "latest_context_path": str(latest_context_path)},
)

print("RUN_ID:", RUN_ID)


RUN_ID: smoke_20260212T200818Z_6b207060


In [3]:
raw_checks = {
    "raw_lspo_parquet": Path(DATA["raw_lspo_parquet"]).exists(),
    "raw_lspo_h5": Path(DATA["raw_lspo_h5"]).exists(),
    "raw_ads_publications": Path(DATA["raw_ads_publications"]).exists(),
    "raw_ads_references": Path(DATA["raw_ads_references"]).exists(),
}
raw_checks


{'raw_lspo_parquet': True,
 'raw_lspo_h5': False,
 'raw_ads_publications': True,
 'raw_ads_references': True}

In [4]:
lspo_mentions = prepare_lspo_mentions(
    parquet_path=DATA["raw_lspo_parquet"],
    h5_path=DATA.get("raw_lspo_h5"),
    output_path=RUN_DIRS["interim"] / "lspo_mentions.parquet",
)
ads_mentions = prepare_ads_mentions(
    publications_path=DATA["raw_ads_publications"],
    references_path=DATA["raw_ads_references"],
    output_path=RUN_DIRS["interim"] / "ads_mentions.parquet",
)

print("LSPO mentions:", len(lspo_mentions))
print("ADS mentions:", len(ads_mentions))


LSPO mentions: 554962
ADS mentions: 1550512


In [5]:
stage = RUN["stage"]
seed = int(RUN.get("seed", 11))
target = RUN.get("subset_target_mentions")

lspo_subset = build_stage_subset(lspo_mentions, stage=stage, seed=seed, target_mentions=target)
ads_subset = build_stage_subset(ads_mentions, stage=stage, seed=seed, target_mentions=target)

lspo_subset_path = RUN_DIRS["subset_cache"] / f"lspo_mentions_{stage}.parquet"
ads_subset_path = RUN_DIRS["subset_cache"] / f"ads_mentions_{stage}.parquet"
lspo_manifest_path = RUN_DIRS["subset_manifests"] / f"{RUN_ID}_lspo_{stage}_manifest.parquet"
ads_manifest_path = RUN_DIRS["subset_manifests"] / f"{RUN_ID}_ads_{stage}_manifest.parquet"

save_parquet(lspo_subset, lspo_subset_path, index=False)
save_parquet(ads_subset, ads_subset_path, index=False)
write_subset_manifest(lspo_subset, lspo_manifest_path)
write_subset_manifest(ads_subset, ads_manifest_path)

print("LSPO subset:", len(lspo_subset), "->", lspo_subset_path)
print("ADS subset:", len(ads_subset), "->", ads_subset_path)


LSPO subset: 1000 -> C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\data\subsets\cache\smoke_20260212T200818Z_6b207060\lspo_mentions_smoke.parquet
ADS subset: 1000 -> C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\data\subsets\cache\smoke_20260212T200818Z_6b207060\ads_mentions_smoke.parquet


In [6]:
def qc_table(df, name):
    return {
        "name": name,
        "records": len(df),
        "mentions": df["mention_id"].nunique(),
        "bibcodes": df["bibcode"].nunique(),
        "blocks": df["block_key"].nunique(),
        "missing_title": int(df["title"].isna().sum() + (df["title"].astype(str).str.strip()=="").sum()),
        "missing_abstract": int(df["abstract"].isna().sum() + (df["abstract"].astype(str).str.strip()=="").sum()),
        "missing_year": int(df["year"].isna().sum()),
        "missing_author": int(df["author_raw"].isna().sum() + (df["author_raw"].astype(str).str.strip()=="").sum()),
    }

pd.DataFrame([
    qc_table(lspo_subset, "lspo_subset"),
    qc_table(ads_subset, "ads_subset"),
])


,name,records,mentions,bibcodes,blocks,missing_title,missing_abstract,missing_year,missing_author
0,lspo_subset,1000,1000,1000,500,0,0,1000,0
1,ads_subset,1000,1000,989,500,0,135,0,0


In [7]:
print("Top-20 ambige LSPO-Blöcke")
display(summarize_block_distribution(lspo_subset, top_k=20))
print("Top-20 ambige ADS-Blöcke")
display(summarize_block_distribution(ads_subset, top_k=20))

def suspect_block_summary(df: pd.DataFrame, name: str) -> dict:
    blocks = df["block_key"].fillna("").astype(str)
    block_df = blocks.to_frame(name="block_key").drop_duplicates()

    surname_len = block_df["block_key"].str.split(".").str[-1].fillna("").str.len()
    short_surname_blocks = int((surname_len <= 2).sum())

    tmp = df[["block_key", "author_raw"]].copy()
    tmp["surname"] = tmp["author_raw"].fillna("").astype(str).str.lower().str.replace(r"[^a-z ]", "", regex=True).str.split().str[-1]
    surname_variants = tmp.groupby("block_key")["surname"].nunique(dropna=True)
    collision_blocks = int((surname_variants >= 4).sum())

    total_blocks = max(1, int(block_df["block_key"].nunique()))
    suspect_blocks = int(((surname_len <= 2).sum()) + collision_blocks)
    return {
        "dataset": name,
        "total_blocks": total_blocks,
        "short_surname_blocks": short_surname_blocks,
        "high_collision_blocks": collision_blocks,
        "suspect_block_ratio": suspect_blocks / total_blocks,
    }

display(pd.DataFrame([
    suspect_block_summary(lspo_subset, "lspo_subset"),
    suspect_block_summary(ads_subset, "ads_subset"),
]))


Top-20 ambige LSPO-Blöcke


,block_key,mention_count
0,a.amorim,2
1,o.guyon,2
2,p.lee,2
3,p.križan,2
4,p.kretschmar,2
5,p.jöckel,2
6,p.fonte,2
7,p.figueira,2
8,p.dunne,2
9,p.chu,2


Top-20 ambige ADS-Blöcke


,block_key,mention_count
0,a.boksenberg,2
1,n.evans,2
2,p.demarque,2
3,p.conti,2
4,p.charles,2
5,p.anderson,2
6,o.m,2
7,o.jb,2
8,o.eggen,2
9,o.e,2


,dataset,total_blocks,short_surname_blocks,high_collision_blocks,suspect_block_ratio
0,lspo_subset,500,30,0,0.060
1,ads_subset,500,32,0,0.064
